Import relevant libraries and import the file using a relative path.
Then Split the data into test and training. We use a specific seed as our random state so that we get the same result every time.

## Imports
- **pandas** is a library for data manipulation. We will use the pandas dataframe as it can easily be turned into PyTorch
- **pathlib** a smart little library used to make relative paths. This way we can have a path that is the same for everyone.
- **torch** The Pytorch library.
- **tranformers** we import AutoConfig and AutoModel from the HugginFace tranformer module. These are used to get the configuration from one of HugginFaces models, and then we create a model from that config using the AutoModel function.
- **train_test_split** this is used to split data into training and testing.

In [32]:
# Data manipulation and visualization libraries
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

# Machine learning libraries
import torch
import torch.nn as nn #neural network module
import torch.nn.functional as F #functional module contains functions that don't have parameters, like activation functions and loss functions
from torch.utils.data import Dataset, DataLoader #Dataset is an abstract class representing a dataset, and DataLoader is a utility that provides an iterable over the given dataset.
from transformers import AutoConfig, AutoModel # AutoConfig is used to load the configuration of a pre-trained model, and AutoModel is used to load the pre-trained model itself.
from sklearn.model_selection import train_test_split #train_test_split is a function from scikit-learn that splits arrays or matrices into random train and test subsets.
from sklearn.preprocessing import StandardScaler # StandsardScaler is a class from scikit-learn that standardizes features by removing the mean and scaling to unit variance.
from sklearn.model_selection import TimeSeriesSplit # TimeSeriesSplit is a class from scikit-learn that provides train/test indices to split time series data samples that are observed at fixed time intervals.
from sklearn.preprocessing import LabelEncoder # LabelEncoder is a class from scikit-learn that encodes target labels with value between 0 and n_classes-1, where n is the number of distinct labels.
from sklearn.metrics import f1_score, classification_report

# Prepare Data
Import the dataset using the path variable. Then use the train_test_split() funtion to split the data into random train and test subsets.

bla bla bla... more text is coming.

## 1. Load & Sort
- Load the data into a dataframe using pandas
- We then use the to_datetime function to make the dates into actual date objectes instead of strings
- We the sort the data by timestamp pr. building

### Optimizations
Because the dataset is so big, some optimisations are needed to free memory
- all numeric values are converted into 32bit variances
- We the change the way category data is stored
    - instead of storing every single string for each row, change the type to category, meaning that it is only stored once, instead of milions of times.
    - **Normal string storage**: pandas stores the full string for every single row
    - **Category storage**: pandas stores each unique string only once in a lookup table, then stores a small integer per row pointing to that table

In [33]:
SEC_PATH = Path("datasets/Smart Grid Security Data/security_dataset.csv")
LEAD_PATH = Path("datasets/LEAD/train_features.csv")
NODE_ID = "building_id"
TIMESTAMP = "timestamp"

df = pd.read_csv(LEAD_PATH) # change path depending on which dataset you want to use

df[TIMESTAMP] = pd.to_datetime(df[TIMESTAMP])
df = df.sort_values(by=[NODE_ID, TIMESTAMP])

# Float64 → float32 (half the memory)
float_cols = df.select_dtypes(include="float64").columns
df[float_cols] = df[float_cols].astype("float32")

# Int64 → int32 (half the memory)
int_cols = df.select_dtypes(include="int64").columns
df[int_cols] = df[int_cols].astype("int32")

# Object (string) → category (much less memory if there are many repeated values)
str_cols = df.select_dtypes(include="str").columns
df[str_cols] = df[str_cols].astype("category")

# Check memory usage after optimization
print(df.info(memory_usage="deep"))

for col in df.select_dtypes(include="str").columns:
    print(f"{col}: {df[col].memory_usage(deep=True) / 1e6:.1f} MB")

<class 'pandas.DataFrame'>
Index: 1749494 entries, 0 to 1749493
Data columns (total 57 columns):
 #   Column                         Dtype         
---  ------                         -----         
 0   building_id                    int32         
 1   timestamp                      datetime64[us]
 2   meter_reading                  float32       
 3   anomaly                        int32         
 4   site_id                        int32         
 5   primary_use                    category      
 6   square_feet                    int32         
 7   year_built                     int32         
 8   floor_count                    int32         
 9   air_temperature                float32       
 10  cloud_coverage                 int32         
 11  dew_temperature                float32       
 12  precip_depth_1_hr              int32         
 13  sea_level_pressure             float32       
 14  wind_direction                 int32         
 15  wind_speed                     

## 2. Features and Target

- Define the features and target of different datasets

In [35]:
LEAD_features = [
    "meter_reading",
    "site_id",
    "square_feet",
    "year_built",
    "floor_count",
    "air_temperature",
    "cloud_coverage",
    "dew_temperature",
    "precip_depth_1_hr",
    "sea_level_pressure",
    "wind_direction",
    "wind_speed",
    "air_temperature_mean_lag7",
    "air_temperature_max_lag7",
    "air_temperature_min_lag7",
    "air_temperature_std_lag7",
    "air_temperature_mean_lag73",
    "air_temperature_max_lag73",
    "air_temperature_min_lag73",
    "air_temperature_std_lag73",
    "hour_x",
    "hour_y",
    "month_x",
    "month_y",
    "weekday_x",
    "weekday_y",
    "is_holiday",
    "meter_lag1",
    "meter_lag24",
    "meter_roll_mean_24",
    "meter_roll_std_24",
    "meter_diff_1",
    "meter_diff_24",
    "meter_zscore_24",
]
LEAD_target = "anomaly"



smart_grid_features = [
    "voltage_level",
    "frequency_signal",
    "power_flow",
    "reactive_power",
    "access_behavior",
    "temporal_entropy",
    "spectral_energy",
    "spatial_correlation",
    "wavelet_coeff_avg",
    "wavelet_coeff_std"
]
smart_grid_target = "attack_type"


## 3. Preprocessing
- Do some preprocessing that is dependent on what dataset we are using
- Categories need to be encoded, and target might also need to be encoded for multiclass classification tasks

### Security Dataset

In [ ]:
# Encode target
le = LabelEncoder()
df["attack_type"] = le.fit_transform(df["attack_type"])

# Encode categorical feature
le_access = LabelEncoder()
df["access_behavior"] = le_access.fit_transform(df["access_behavior"])

# Handle missing values
feature_cols = smart_grid_features
df = df.dropna()

### The LEAD Dataset

#### Pre-building Lag Features
- Lag features are past values of a variable.
- We bring this value forward to the current timestep
- We do this because we want to be able to memorize all past timesteps
- **Example**: `meter_lag1`means what was the meter reading 1 hour ago
- Because we group by building_id, there is no data leakage from other buildings
- `.transform()` guarantees that the output has the **same shape and index as the input**, so the result slots back into `LEAD_df` correctly, one value per row, aligned to the right building and timestamp.
- the lambda x part is how you zmake lambda functions in python

- **meter_diff_1**: change in the last hour. 
    - Catches sudden spikes or drops.
    - A jump from 100 to 950 in one hour is a strong anomaly signal that the raw reading alone would not reveal.
- **meter_diff_24**: change since the same hour yesterday.
    -Captures slower, day-over-day shifts.
    - A building that normally uses 200 kWh on Monday mornings but suddenly uses 800 kWh is suspicious, even if the change happened gradually over the hour.
- **meter_zscore_24**: how many standard deviations the current reading is from the 24-hour mean 
    - Normalises the signal across buildings.
    - A reading of 500 kWh might be completely normal for a large office block but extreme for a small retail unit.

In [37]:
# These must be computed per building (via groupby) and BEFORE the train/test
# split — they are feature engineering, not data leakage, because each value
# only looks backwards in time within its own building.

groups = df.groupby(NODE_ID)  # group the data by node so we can compute features separately for each node

# copy a past value into the current row so the model can see history.
df["meter_lag1"]  = groups["meter_reading"].shift(1)  #what was the meter reading 1 hour ago?
df["meter_lag24"] = groups["meter_reading"].shift(24) #what was the meter reading 24 hours ago?


# Rolling mean over the last 24 hours — captures the building's "normal" baseline.
# min_periods=1 means it still produces a value even near the start of the series.
df["meter_roll_mean_24"] = groups["meter_reading"].transform(
    lambda x: x.rolling(window=24, min_periods=1).mean()
)

# Rolling std over the last 24 hours — captures how volatile the recent period was.
# A low std means stable consumption; a high std means erratic behaviour.
df["meter_roll_std_24"] = groups["meter_reading"].transform(
    lambda x: x.rolling(window=24, min_periods=1).std()
)

# Differences — how much has consumption changed since N steps ago?
# We add new columns for the change since 1 hour ago and since 24 hours ago.
df["meter_diff_1"]  = df["meter_reading"] - df["meter_lag1"]   # change in last hour
df["meter_diff_24"] = df["meter_reading"] - df["meter_lag24"]  # change since yesterday

# Z-score — how many standard deviations the current reading is from the 24-hour mean.
# e.g. zscore=0.3 → normal, zscore=7.0 → very likely anomalous.
# +1e-6 avoids division by zero when std is 0 (flat signal with no variation).
df["meter_zscore_24"] = (
    (df["meter_reading"] - df["meter_roll_mean_24"])
    / (df["meter_roll_std_24"] + 1e-6)
)

#### One-hot Encoding
- We expliitly write what columns we need.
- We use one-hot encoding to encode non-numeric values.
    - This works by converting the categories into a binary representation.
    - Each row has exactly one 1, a building can only have one primary use, so only one of these columns is ever 1 per row, the rest are 0.

- **Example**:

```
        building_id   primary_use
        1             Office
        2             Retail
        3             Office
        4             Education
        5             Retail

        building_id   primary_use_Retail   primary_use_Education
        1             0                    0
        2             1                    0
        3             0                    0
        4             0                    1
        5             1                    0
```
- In case we get NaN values, we drop them.


In [38]:
df = pd.get_dummies(df, columns=["primary_use"], drop_first=True) # one-hot encoding
df = df.dropna() # drop rows with NaN values

primary_use_cols = []
for col in df.columns:
    if col.startswith("primary_use_"):
        primary_use_cols.append(col)

feature_cols = LEAD_features + primary_use_cols

X = df[feature_cols] # the features (input variables) for the model
y = df[LEAD_target] # the target variable (what we want to predict)

## 4. Sliding Window Helper
- Given the last `n` hours of data, predict whether the next hour is an anormaly

- Converts a single building's flat time series into sliding windows.
- For each position `i`, it collects:
    - X: the rows from `i` to `i+window_size`  (the look-back features)
    - y: the anomaly label at `i+window_size` (the target to predict)

In [39]:
def create_windowed_data(df, feature_cols, window_size, stride, target):

    X_windows, y_windows = [], []
    data   = df[feature_cols].values   # shape: (n_rows, n_features)
    labels = df[target].values         # shape: (n_rows,)

    # range(start, stop, step):
    #   start = 0               → begin at first row
    #   stop  = len-window_size → last valid start so window doesn't fall off the end
    #   step  = stride          → how far to shift each iteration
    for i in range(0, len(data) - window_size, stride):
        X_windows.append(data[i : i + window_size])   # rows i..i+window_size-1
        y_windows.append(labels[i + window_size])     # label just after the window

    return np.array(X_windows), np.array(y_windows)


## 5. Temporal Grouped Train/Test Split
Splits a multi-building time series dataset into train and test windows while respecting both temporal order and building groups.

- Find a single global cutoff timestamp at the train_ratio percentile of all timestamps, so every building shares the same calendar boundary.

- For each building, split its rows into train (before cutoff) and test (after cutoff + gap).

- Apply sliding windows separately to train and test portions.

The gap prevents look-back leakage: the longes lag feature is 73 hours,
so the first test window must not be able to "see back" into training data
via lagged features.

In [40]:
def temporal_grouped_split(
    df,
    feature_cols,
    window_size,
    stride,
    node_col,
    time_col="timestamp",
    train_ratio=0.8,
    gap_hours = 0, # In case of no lag features, set gap_hours=0.
    target="anomaly",
):
    """
    Parameters
    ----------
    df           : pre-processed DataFrame
    feature_cols : list of feature column names
    node_col     : column name for node identifier
    time_col     : column name for timestamp
    train_ratio  : fraction of time to use for training (e.g. 0.8 = 80%)
    gap_hours    : hours to skip between train end and test start
                   (For LEAD, use at least max lag = 73 to be safe)
    window_size  : look-back window length in timesteps 
    stride       : window shift per iteration (24 = one window per day)

    Returns
    -------
    X_train, y_train : training windows and labels
    X_test,  y_test  : test windows and labels
    nid_train        : node_id for each training window (useful for analysis)
    nid_test         : node_id for each test window
    """
    df = df.sort_values([node_col, time_col])

    # Single global cutoff — the timestamp at the train_ratio position
    # across ALL rows (all buildings combined), sorted by time.
    # iloc[] is used because we need positional indexing, not label indexing.
    all_times = df[time_col].sort_values()
    cutoff    = all_times.iloc[int(len(all_times) * train_ratio)]

    X_train_list, y_train_list = [], []
    X_test_list,  y_test_list  = [], []
    nid_train, nid_test        = [], []

    for node, group in df.groupby(node_col):
        group = group.sort_values(time_col)

        # Split at cutoff, with a gap after cutoff to avoid lag leakage
        train_df = group[group[time_col] <= cutoff]
        test_df  = group[group[time_col] >  cutoff + pd.Timedelta(hours=gap_hours)] # use pandas' Timedelta class to add hours to a timestamp

        # Only proceed if the split has enough rows for at least one full window
        if len(train_df) > window_size:
            Xtr, ytr = create_windowed_data(train_df, feature_cols, window_size, stride, target)
            X_train_list.append(Xtr)
            y_train_list.append(ytr)
            nid_train.extend([node] * len(Xtr))

        if len(test_df) > window_size:
            Xte, yte = create_windowed_data(test_df, feature_cols, window_size, stride, target)
            X_test_list.append(Xte)
            y_test_list.append(yte)
            nid_test.extend([node] * len(Xte))
    
    # Concatenate all windows from all buildings into single arrays for train and test sets.
    X_train = np.concatenate(X_train_list, axis=0)
    y_train = np.concatenate(y_train_list, axis=0)
    X_test  = np.concatenate(X_test_list,  axis=0)
    y_test  = np.concatenate(y_test_list,  axis=0)

    return X_train, y_train, X_test, y_test, np.array(nid_train), np.array(nid_test)

#### Run this BEFORE the split to estimate memory usage

In [41]:
total_rows = len(df)
approx_windows = total_rows / 24  # stride=24
window_size = 168
n_features = len(feature_cols)

memory_gb = (approx_windows * window_size * n_features * 4) / 1e9  # float32 = 4 bytes
print(f"Approximate number of windows: {approx_windows:,.0f}")
print(f"Estimated memory for X_train alone: {memory_gb:.1f} GB")

Approximate number of windows: 67,657
Estimated memory for X_train alone: 2.0 GB


### Run The Split

In [ ]:
# SECURITY DATASET
X_train, y_train, X_test, y_test, nid_train, nid_test = temporal_grouped_split(
    df,
    feature_cols,
    node_col="node_id",
    time_col="timestamp",       
    train_ratio=0.6,
    gap_hours=0,               
    window_size=8,         
    stride=4,     
    target=smart_grid_target,            
)

In [43]:
# LEAD DATASET
X_train, y_train, X_test, y_test, nid_train, nid_test = temporal_grouped_split(
    df,
    feature_cols,
    node_col="building_id",
    time_col="timestamp",       
    train_ratio=0.8,
    gap_hours=73,               # matches longest lag feature (lag73)
    window_size=168,            # 1 week of hourly data
    stride=24,                  # one window per day
    target=LEAD_target,
)

### 6. Scaling
This might take a while, and use a lot of ram

In [ ]:

# IMPORTANT: fit the scaler ONLY on training data, then apply to both.
# Per-building scaling done in-place to minimize memory usage.

num_train, num_timesteps, num_features = X_train.shape

# Convert to float32 in-place if not already
X_train = X_train.astype(np.float32, copy=False)
X_test  = X_test.astype(np.float32, copy=False)

# Fit one scaler per building on training windows, transform in-place
scalers = {}
for node in np.unique(nid_train):
    mask = nid_train == node
    scaler = StandardScaler()
    flat = X_train[mask].reshape(-1, num_features)
    scaler.fit(flat)
    X_train[mask] = scaler.transform(flat).reshape(-1, num_timesteps, num_features)
    scalers[node] = scaler
    del flat

# Apply each building's scaler to its test windows in-place
# If a building appears in test but not train, use a neighbor or skip — see count below
fallback_count = 0
for node in np.unique(nid_test):
    mask = nid_test == node
    if node in scalers:
        flat = X_test[mask].reshape(-1, num_features)
        X_test[mask] = scalers[node].transform(flat).reshape(-1, num_timesteps, num_features)
        del flat
    else:
        # Rare fallback: fit a scaler on this building's own test data
        # (slight leak, but only affects ~1 building out of 200)
        fallback_count += 1
        scaler = StandardScaler()
        flat = X_test[mask].reshape(-1, num_features)
        X_test[mask] = scaler.fit_transform(flat).reshape(-1, num_timesteps, num_features)
        del flat

# Use the in-place arrays as the scaled versions
X_train_scaled = X_train
X_test_scaled  = X_test

print(f"X_train : {X_train_scaled.shape}")
print(f"y_train : {y_train.shape}")
print(f"X_test  : {X_test_scaled.shape}")
print(f"y_test  : {y_test.shape}")
print(f"Train nodes : {np.unique(nid_train).size}")
print(f"Test  nodes : {np.unique(nid_test).size}")
print(f"Buildings with own scaler: {len(scalers)}")
print(f"Test buildings using fallback: {fallback_count}")
print(f"Anomaly rate train: {y_train.mean():.3f}")
print(f"Anomaly rate test : {y_test.mean():.3f}")

# Device Helper Function

In [ ]:
def get_device():
    """
    Returns the best available device in priority order:
      1. ROCm (AMD GPU via HIP) — detected through torch.cuda, which ROCm mirrors
      2. CUDA (Nvidia GPU)      — same API, included for completeness
      3. CPU                    — fallback if no GPU is available
    """
    if torch.cuda.is_available():
        device = torch.device("cuda")
        gpu_name = torch.cuda.get_device_name(0)
        print(f"GPU available: {gpu_name}")
    else:
        device = torch.device("cpu")
        print("No GPU found, using CPU")
    
    return device

device = get_device()

print(torch.cuda.is_available())

# Plotting features

This is how to plot features using matplotlib.
Might become valuable if we want to analyse specific features later.

In [ ]:
#Hvordan man plotter en feature? (Måske er det nydvendigt senere)
df_numeric = df.apply(pd.to_numeric, errors="coerce")

# Plot Specific features using regex to filter columns that match the pattern 'R1-PA[1-3]:VH'
plt.plot(df_numeric.filter(['power_flow']))
plt.xlabel("Sample index")
plt.ylabel("Feature value")
plt.title("Power Flow over samples")
plt.show()

# Plot features using iloc to select specific columns for the first 100 samples
plt.plot(df_numeric.iloc[1:100, 1:5])
plt.xlabel("Sample index")
plt.ylabel("Feature value")
plt.title("First 5 Features over samples")
plt.show()

# CNN Feature Extractor

- self.cnn = nn.Sequential()
    - order of inputs define which order each layer should execute in

- nn.Conv1d(in_channels, 32, kernel_size=3, stride=2, padding=1)
    - in_channels : Number of input features, so how many variables are in each timestep
    - 32 : Output channels. How many patterns the CNN will learn.
    - Kernel_size : Size of the filter
    - Stride : how many steps we move the filter
    - padding : how many 0 we add to the start and end of the input. Prevents the sequence from shrinking too much. 

- nn.BatchNorm1d(32)
    - Normalizes the activations 
    
- nn.ReLU(inplace=True)
    - Means we are using relu
    - Inplace : modifys the current tensor directly, instead of creating a whole new tensor. This saves memory. 

- nn.MaxPool1d(2)
    - Reduce the sequence lenght by only keeping only the strongest activations
    - the 2, means that the seqence lenght is halved

In [19]:
class FeatureExtractor(nn.Module):
    def __init__(self, in_channels, d_model, dropout=0.3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2), # reduce the sequence length by half
            nn.Dropout1d(dropout),

            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2), 
            nn.Dropout1d(dropout),

            nn.Conv1d(128, d_model, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(d_model),
            nn.ReLU()
        )
        
    def forward(self, x):
        return self.features(x)

# Classifier
This module implements the final classification head of the model. Its purpose is to thransform the feature representation produced by the FeatureExtractor into class logits.

The classifier takes a feature vectore and predicts the probability of each class.

- Flatten
    - Converts the input tensor into a 1-dimentional feature vector
- Linear Layer
    - Fully connected layer that learns combinations of the extracted features.
    - Reduces the feature dimension
- ReLU
    - Applies the non-linear function `f(x) = max(0, x)`
- Dropout
    - Randomly disables 20% of neurons during training.
    - Helps prevent overfitting by making the model rely on multiple features instead of a few dominant ones
- Linear Layer
    - Produces the final logits for each class.
    - For binary classification (`num_classes = 2`), the output shape becomes: `(batch_size, 2)`

In [20]:
class Classifier(nn.Module):
    def __init__(self, d_model, num_classes=1, dropout=0.4):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes) # output for multi-class classification
        )

    def forward(self, x):
        return self.classifier(x)

# Positional Encoder

In [21]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)

        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (B, T, d_model)
        return x + self.pe[:, :x.size(1)]

# CNNTRansformer

- num_classes : Number of classes we want to classify prediction in. In our case it is binary (True / False)

- in_channels : Number of columns in dataset, or number of culumns the model should analyze. In our case it is all columns, expect for timestep. According to chat, we don't need timestep to make preidiction. 

- embed_dim : size of feature vector used by transformer. if training is unstable, reduce to 64, if model overfits, use 256. Embed_dim must be divisable by num_heads (FInd ud af specifikt h)

- num_heads : Number of attention heads in transformers mult-head attention layer. 

- num_layers : Number of transformer layers (?)

- mlp_dim : Size of feedforward inside each transformor layer (Hvorfor er det 256)

- dropout : Percentage of neurons we randomly turn off during each training step. 

In [22]:
class CNNTransformer(nn.Module):
    def __init__(self, in_channels, d_model, nhead, num_layers, num_classes=1, dropout=0.3):
        super().__init__()
        
        self.feature_extractor = FeatureExtractor(in_channels, d_model, dropout=dropout)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder( encoder_layer, num_layers=num_layers)

        self.classifier = Classifier(d_model, num_classes, dropout=dropout)

    def forward(self, x):
        # x: (batch_size, seq_len, in_channels)
        x = x.permute(0, 2, 1)                    # (batch_size, in_channels, seq_len)

        features = self.feature_extractor(x)      # (batch_size, d_model, seq_len//4)
        features = features.permute(0, 2, 1)      # (batch_size, seq_len//4, d_model)

        features = self.pos_encoder(features)      # add positional encoding
        transformer_out = self.transformer_encoder(features)   # (batch_size, seq_len//4, d_model)

        out = transformer_out.mean(dim=1)         # (batch_size, d_model)
        out = self.classifier(out)                # (batch_size, num_classes)

        return out.squeeze(-1)

# Train and Validate the Model

In [23]:
from torch.utils.data import TensorDataset, DataLoader

def build_dataloader(features, labels, batch_size=64, shuffle=False, num_classes=1):
    feature_tensor = torch.tensor(features, dtype=torch.float32)
    label_tensor = torch.tensor(labels, dtype=torch.long if num_classes > 1 else torch.float32)
    dataset = TensorDataset(feature_tensor, label_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def run_train_and_validation_epoch(
    model,
    train_dataloader,
    validation_dataloader,
    optimizer,
    device,
    loss_fn,
    num_classes=1,

):

    model.train() # Train function from nn.Module sets the model to training mode (enables dropout, batchnorm updates, etc.)
    total_train_loss, total_train_correct, total_train_samples = 0.0, 0, 0

    for batch_features, batch_labels in train_dataloader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        optimizer.zero_grad()
        logits = model(batch_features)
        loss = loss_fn(logits, batch_labels)
        loss.backward()
        optimizer.step()

        if num_classes == 1:
            predictions = (torch.sigmoid(logits) >= 0.5).float()
        else:
            predictions = logits.argmax(dim=1)
        total_train_loss += loss.item() * batch_features.size(0)
        total_train_correct += (predictions == batch_labels).sum().item()
        total_train_samples += batch_features.size(0)


    model.eval() # Eval function from nn.Module sets the model to evaluation mode (disables dropout, batchnorm updates, etc.)
    total_val_loss, total_val_samples = 0.0, 0
    all_preds, all_labels = [], []   # ← accumulate here for F1 score

    with torch.no_grad():
        for batch_features, batch_labels in validation_dataloader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            logits = model(batch_features)
            loss = loss_fn(logits, batch_labels)

            if num_classes == 1:
                predictions = (torch.sigmoid(logits) >= 0.5).float()
            else:
                predictions = logits.argmax(dim=1)

            total_val_loss    += loss.item() * batch_features.size(0)
            total_val_samples += batch_features.size(0)

            all_preds.append(predictions.cpu())    # ← move to CPU and store
            all_labels.append(batch_labels.cpu())  # ← move to CPU and store

    # Concatenate all batches into single arrays
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    val_f1 = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)
    # zero_division=0 means: if the model predicts no anomalies at all, return F1=0
    # instead of throwing a warning — useful in early epochs when the model is still collapsed

    return {
        "train_loss":        total_train_loss / total_train_samples,
        "train_accuracy":    total_train_correct / total_train_samples,
        "validation_loss":   total_val_loss / total_val_samples,
        "validation_f1":     val_f1,                          # ← replaces val accuracy
    }

## Train function

In [ ]:
def train(model, X_train, y_train, X_val, y_val, loss_fn, num_classes, epochs, batch_size, lr):
    device = get_device()
    model = model.to(device)

    train_dataloader = build_dataloader(X_train, y_train, batch_size=batch_size, shuffle=True,  num_classes=num_classes)
    val_dataloader   = build_dataloader(X_val,   y_val,   batch_size=batch_size, shuffle=False, num_classes=num_classes)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=6.14e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # Early stopping state
    best_f1 = -1.0
    best_state = None
    patience = 7
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        metrics = run_train_and_validation_epoch(
            model, train_dataloader, val_dataloader, optimizer, device, loss_fn=loss_fn, num_classes=num_classes
        )
        scheduler.step()

        val_f1 = metrics['validation_f1']
        print(
            f"Epoch {epoch:>3}/{epochs} | "
            f"Train loss: {metrics['train_loss']:.4f}  acc: {metrics['train_accuracy']:.4f} | "
            f"Val loss: {metrics['validation_loss']:.4f}  f1: {val_f1:.4f}  "
            f"lr: {scheduler.get_last_lr()[0]:.2e}"
        )

        # Early stopping: track best val F1 and restore if no improvement
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch} (best val F1: {best_f1:.4f})")
                break

    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best model (val F1: {best_f1:.4f})")

    # Debug info
    batch_features, batch_labels = next(iter(train_dataloader))
    print(f"Feature dtype: {batch_features.dtype}")
    print(f"Label dtype:   {batch_labels.dtype}")
    print(f"Feature shape: {batch_features.shape}")
    print(f"Any NaN in batch: {torch.isnan(batch_features).any()}")
    print(f"Label unique: {batch_labels.unique()}")

    model.eval()
    with torch.no_grad():
        x = batch_features.to(device)
        print("Input variance:", x.var().item())
        
        x_perm = x.permute(0, 2, 1)
        cnn_out = model.feature_extractor(x_perm)
        print("CNN output variance:", cnn_out.var().item())
        
        cnn_perm = cnn_out.permute(0, 2, 1)
        pos_out = model.pos_encoder(cnn_perm)
        print("After pos encoding variance:", pos_out.var().item())
        
        trans_out = model.transformer_encoder(pos_out)
        print("Transformer output variance:", trans_out.var().item())
        
        pooled = trans_out.mean(dim=1)
        print("After pooling variance:", pooled.var().item())

    return model

### Start the training

In [27]:
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
pos_weight = min(pos_weight, 7.01)  # cap to avoid over-correcting on rare positives
print(f"pos_weight: {pos_weight:.2f}")
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))

model = CNNTransformer(
    in_channels=X_train_scaled.shape[2],
    d_model=128,
    nhead=4,
    num_layers=3,
    num_classes=1
).to(device)
model = train(
    model,
    X_train_scaled,
    y_train,
    X_test_scaled, 
    y_test, 
    loss_fn=loss_fn, 
    num_classes=1, 
    epochs=30, 
    batch_size=32, 
    lr=1e-4)

pos_weight: 7.01
GPU available: AMD Radeon Graphics
Epoch   1/30 | Train loss: 0.4139  acc: 0.9696 | Val loss: 0.3198  f1: 0.4029  lr: 9.97e-05
Epoch   2/30 | Train loss: 0.3128  acc: 0.9737 | Val loss: 0.3199  f1: 0.3813  lr: 9.89e-05
Epoch   3/30 | Train loss: 0.2886  acc: 0.9734 | Val loss: 0.3350  f1: 0.3700  lr: 9.76e-05
Epoch   4/30 | Train loss: 0.2745  acc: 0.9750 | Val loss: 0.3523  f1: 0.4172  lr: 9.57e-05


KeyboardInterrupt: 

In [ ]:
print(f"NaNs in X_train_scaled: {np.isnan(X_train_scaled).sum()}")
print(f"Infs in X_train_scaled: {np.isinf(X_train_scaled).sum()}")
print(f"NaNs in y_train: {np.isnan(y_train.astype(float)).sum()}")

## Optuna Hyperparameter Search
Searches for the best combination of hyperparameters by maximizing validation F1.

In [26]:
import optuna

def objective(trial):
    # Sample hyperparameters
    d_model = trial.suggest_categorical("d_model", [32, 64, 128])
    nhead = trial.suggest_categorical("nhead", [2, 4])
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.2, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    pos_weight_cap = trial.suggest_float("pos_weight_cap", 5.0, 20.0)

    device = get_device()

    # Build loss with capped pos_weight
    raw_pw = (y_train == 0).sum() / (y_train == 1).sum()
    pw = min(float(raw_pw), pos_weight_cap)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))

    # Build model
    model = CNNTransformer(
        in_channels=X_train_scaled.shape[2],
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        num_classes=1,
        dropout=dropout,
    ).to(device)

    # Build dataloaders
    train_dl = build_dataloader(X_train_scaled, y_train, batch_size=batch_size, shuffle=True, num_classes=1)
    val_dl = build_dataloader(X_test_scaled, y_test, batch_size=batch_size, shuffle=False, num_classes=1)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

    best_f1 = 0.0
    patience = 7
    no_improve = 0

    for epoch in range(1, 31):
        metrics = run_train_and_validation_epoch(
            model, train_dl, val_dl, optimizer, device, loss_fn=loss_fn, num_classes=1
        )
        scheduler.step()

        val_f1 = metrics["validation_f1"]

        # Report to Optuna for pruning
        trial.report(val_f1, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_f1 > best_f1:
            best_f1 = val_f1
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    return best_f1

study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
study.optimize(objective, n_trials=20)

print("\n=== Best Trial ===")
print(f"  F1:    {study.best_trial.value:.4f}")
print(f"  Params:")
for k, v in study.best_trial.params.items():
    print(f"    {k}: {v}")

[I 2026-03-30 23:04:41,708] A new study created in memory with name: no-name-4e313407-82d8-48ab-916e-885fd3a011d6


GPU available: AMD Radeon Graphics


[I 2026-03-30 23:05:22,536] Trial 0 finished with value: 0.0 and parameters: {'d_model': 32, 'nhead': 2, 'num_layers': 2, 'dropout': 0.27565843272019525, 'lr': 0.0018716378866779288, 'weight_decay': 0.00033777347191004303, 'batch_size': 64, 'pos_weight_cap': 16.014271748123058}. Best is trial 0 with value: 0.0.


GPU available: AMD Radeon Graphics


[I 2026-03-30 23:06:01,686] Trial 1 finished with value: 0.35233160621761656 and parameters: {'d_model': 32, 'nhead': 4, 'num_layers': 1, 'dropout': 0.2909108948597365, 'lr': 0.0008017751675589392, 'weight_decay': 0.001402229895678902, 'batch_size': 128, 'pos_weight_cap': 10.185314212401611}. Best is trial 1 with value: 0.35233160621761656.


...

GPU available: AMD Radeon Graphics


[I 2026-03-30 23:33:37,403] Trial 18 pruned. 


GPU available: AMD Radeon Graphics


[I 2026-03-30 23:34:35,355] Trial 19 pruned. 



=== Best Trial ===
  F1:    0.4877
  Params:
    d_model: 128
    nhead: 4
    num_layers: 3
    dropout: 0.40516632264878893
    lr: 0.00017727452897133591
    weight_decay: 0.0061425683458801025
    batch_size: 32
    pos_weight_cap: 7.016436429468839
